# Where the data lives

Could we make SYKE's water quality map ourselves?

1. A national dataset read straight from Roihu's disk
2. Finding Sentinel-2 scenes with a catalogue search (STAC)
3. Opening our scene over the network without downloading it
4. A first water quality map, NDTI and turbidity from the same image

All the code is in `eo_tools.py`. Section 1 works only on Roihu.

In [ ]:
import eo_tools as eo

## 1. The NLS 10 m elevation model, read from the Roihu disk

In [ ]:
dem = eo.read_dem()
eo.plot_dem_latlon(dem, eo.VAASA_TM35, crs="EPSG:3067",
                   title="Vaasa and the Kvarken coast, NLS 10 m DEM")

## 2. Which Sentinel-2 scenes exist over Vaasa? (Paituli STAC)

In [ ]:
scenes = eo.find_scenes(start="2026-06-01", end="2026-08-31", max_cloud=10)
print(len(scenes), "scenes with at most 10 % cloud")
scenes

## 3. Open our scene

In [ ]:
item = eo.get_scene(eo.SCENE_ID, table=scenes)
print("Reading from", item.assets["TCI_60m"].href[:60], "...")

import matplotlib.pyplot as plt
tile = eo.read_tci(item, bounds=None, res="60m") 
plt.figure(figsize=(7, 7)); plt.imshow(tile); plt.axis("off")
plt.title("Sentinel-2 tile 34VER, 21 June 2026, 60 m"); plt.show()

In [ ]:
tci = eo.read_tci(item)                                  
plt.figure(figsize=(7, 7)); plt.imshow(tci); plt.axis("off")
plt.title("Same scene, 10 m, Vaasa"); plt.show()

## 4. From image to map, NDTI and turbidity

In [ ]:
green = eo.read_band(item, "B03_10m")
red   = eo.read_band(item, "B04_10m")
nir   = eo.read_band(item, "B08_10m")

water = eo.water_mask(green, red, nir)
ndti  = eo.ndti(red, green)
turb  = eo.turbidity_nechad(red, water)

eo.plot_water_quality(tci, ndti, turb, water, title="Sentinel-2 L2A, Vaasa, 21 June 2026")

for name, v in eo.value_at_sites(ndti).items():
    print(f"{name:22s} NDTI {v:6.2f}   turbidity {eo.value_at_sites(turb)[name]:6.1f} FNU")